# Lotto Modeling Suite

This notebook is responsible for model construction and experiment generation.

Models included:
1. Frequency heuristic baseline
2. Gap heuristic baseline
3. Random baseline
4. Logistic regression
5. Random forest
6. XGBoost
7. Classifier chain

The notebook trains or generates all model outputs, runs the rolling backtest, and saves reusable evaluation artifacts for the next notebook.

### 1. Library Imports

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parent))

import pandas as pd

from src.config import MODEL_OUTPUT_DIR
from src.models.model_suite import run_full_model_suite
from src.visualization import save_report_table


### 2. Experiment Configuration

Adjust these values if you want to change the feature window, holdout split size, execution mode, or rolling backtest setup.

Use `RUN_MODE = "quick"` for a fast iteration loop.
Use `RUN_MODE = "full"` to keep all holdout models while still using a lighter backtest by default.

In [2]:
RUN_MODE = "full"

WINDOW = 20
TEST_RATIO = 0.2
RANDOM_SEED = 42

BACKTEST_INITIAL_TRAIN_SIZE = 600
BACKTEST_TEST_SIZE = 30
BACKTEST_STEP_SIZE = 30

# Quick mode keeps the notebook responsive while you validate the pipeline.
# Full mode still evaluates all holdout models, but the rolling backtest stays
# lighter by default because it skips the heaviest models.
if RUN_MODE == "quick":
    HOLDOUT_MODEL_NAMES = [
        "freq_heuristic",
        "gap_heuristic",
        "random_baseline",
        "logistic_regression",
    ]
    BACKTEST_MODEL_NAMES = [
        "freq_heuristic",
        "gap_heuristic",
        "random_baseline",
        "logistic_regression",
    ]
    INCLUDE_BACKTEST = False
    MAX_BACKTEST_FOLDS = None
elif RUN_MODE == "full":
    HOLDOUT_MODEL_NAMES = [
        "freq_heuristic",
        "gap_heuristic",
        "random_baseline",
        "logistic_regression",
        "random_forest",
        "xgboost",
        "classifier_chain",
    ]
    BACKTEST_MODEL_NAMES = [
        "freq_heuristic",
        "gap_heuristic",
        "random_baseline",
        "logistic_regression",
    ]
    INCLUDE_BACKTEST = True
    MAX_BACKTEST_FOLDS = None
else:
    raise ValueError("RUN_MODE must be either 'quick' or 'full'.")

# If you want to override the presets above, edit the variables below directly.
# Example: BACKTEST_MODEL_NAMES = HOLDOUT_MODEL_NAMES
# Example: MAX_BACKTEST_FOLDS = 6

### 3. Run the Modeling Suite

This cell prepares the aligned dataset, runs the selected holdout models, optionally computes the rolling backtest, and saves the outputs under the model output directory.

In [3]:
results = run_full_model_suite(
    window=WINDOW,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
    backtest_initial_train_size=BACKTEST_INITIAL_TRAIN_SIZE,
    backtest_test_size=BACKTEST_TEST_SIZE,
    backtest_step_size=BACKTEST_STEP_SIZE,
    holdout_model_names=HOLDOUT_MODEL_NAMES,
    backtest_model_names=BACKTEST_MODEL_NAMES,
    include_backtest=INCLUDE_BACKTEST,
    max_backtest_folds=MAX_BACKTEST_FOLDS,
)

results["metadata"]

{'window': 20,
 'test_ratio': 0.2,
 'random_seed': 42,
 'holdout_model_names': ['freq_heuristic',
  'gap_heuristic',
  'random_baseline',
  'logistic_regression',
  'random_forest',
  'xgboost',
  'classifier_chain'],
 'backtest_model_names': ['freq_heuristic',
  'gap_heuristic',
  'random_baseline',
  'logistic_regression'],
 'include_backtest': True,
 'backtest_initial_train_size': 600,
 'backtest_test_size': 30,
 'backtest_step_size': 30,
 'max_backtest_folds': None,
 'n_rows': 1197,
 'n_train_rows': 957,
 'n_test_rows': 240}

### 4. Saved Holdout Summary

In [4]:
results["holdout_summary"]

,model,subset_accuracy,number_level_accuracy,avg_hit,hit_std
6,classifier_chain,0.0,0.773333,0.900000,0.830662
3,logistic_regression,0.0,0.772037,0.870833,0.824105
2,random_baseline,0.0,0.770556,0.837500,0.709056
5,xgboost,0.0,0.770185,0.829167,0.851459
0,freq_heuristic,0.0,0.767037,0.758333,0.763717
4,random_forest,0.0,0.766481,0.745833,0.799729
1,gap_heuristic,0.0,0.765370,0.720833,0.742450


### 5. Saved Rolling Backtest Summary

In [5]:
results["backtest_summary"]

,model,folds,mean_subset_accuracy,mean_number_level_accuracy,mean_avg_hit,std_avg_hit
3,random_baseline,19,0.0,0.772086,0.871930,0.150005
2,logistic_regression,19,0.0,0.771930,0.868421,0.156928
1,gap_heuristic,19,0.0,0.769123,0.805263,0.181986
0,freq_heuristic,19,0.0,0.768109,0.782456,0.141145


### 6. Output Files

The next notebook reads these saved artifacts instead of retraining all models again.

In [6]:
pd.Series({key: str(value) for key, value in results["paths"].items()})

holdout_summary                     /workspace/models/results/holdout_summary.csv
holdout_draw_results            /workspace/models/results/holdout_draw_results...
backtest_results                   /workspace/models/results/backtest_results.csv
backtest_summary                   /workspace/models/results/backtest_summary.csv
metadata                              /workspace/models/results/run_metadata.json
artifact_logistic_regression    /workspace/models/artifacts/logistic_regressio...
artifact_random_forest           /workspace/models/artifacts/random_forest.joblib
artifact_xgboost                       /workspace/models/artifacts/xgboost.joblib
artifact_classifier_chain       /workspace/models/artifacts/classifier_chain.j...
dtype: object

### 7. Summary

This notebook now focuses on modeling only.

It prepares the aligned dataset, runs the selected baselines and machine-learning models, optionally executes the rolling backtest, and saves the holdout and backtest outputs as reusable files.

The next notebook is dedicated to evaluation, comparison, visualization, and interpretation of these saved results.

### Report Export


In [7]:
# Save key model outputs for the final report
save_report_table(results["holdout_summary"], "table_05_holdout_summary.csv")
save_report_table(results["backtest_summary"], "table_06_backtest_summary.csv")
save_report_table(pd.DataFrame([results["metadata"]]), "table_08_run_metadata.csv")
save_report_table(pd.Series({key: str(value) for key, value in results["paths"].items()}).rename_axis("artifact").reset_index(name="path"), "table_09_model_output_paths.csv")
print("Saved model baseline report artifacts.")


Saved model baseline report artifacts.
